# Scripted First-Level Analyses in FSL using fMRIPrep data

This notebook walks through a simple, novice-friendly workflow in **Neurodesk EDU**:

1. install one OpenNeuro dataset and get one subject  
2. run **fMRIPrep** for that subject and extract FEAT-ready confounds  
3. convert BIDS `events.tsv` files to FSL 3-column EV files  
4. render and run a first-level FEAT model from a trust-game FEAT template  
5. make a first pass at viewing the result in **NiiVue**

> **Disclosure / authorship note:** This notebook was generated with assistance from **ChatGPT-5** across several iterations and then revised by the instructor. The instructor reviewed the final content and takes responsibility for it.


**Author**: David V. Smith  

**Date**: March 28, 2026  

**License:** 
<div style="margin-top: 10px;">
    <a href="https://opensource.org/licenses/MIT" target="_blank" style="color: #0066cc;">
        <i class="fas fa-balance-scale"></i> MIT License
    </a>
</div>

**Provenance:** This notebook adapts course materials and selected logic from the Fareri trust-game workflow code, but rewrites the steps so they can be followed directly in notebook cells. Full workflow repository: `https://github.com/tubric/fareri-2022-neuroimage`.


### Citation and Resources

#### Study-specific references
- **Fareri et al. (2022)**: Fareri, D. S., Hackett, K., Tepfer, L. J., Kelly, V., Henninger, N., Reeck, C., Giovannetti, T., & Smith, D. V. (2022). *Age-related differences in ventral striatal and default mode network function during reciprocated trust.* **NeuroImage, 256**, 119267. https://doi.org/10.1016/j.neuroimage.2022.119267
- **Smith et al. (2024)**: Smith, D. V., Ludwig, R. M., Dennison, J. B., Reeck, C., & Fareri, D. S. (2024). *An fMRI Dataset on Social Reward Processing and Decision Making in Younger and Older Adults.* **Scientific Data, 11**(1), 158. https://doi.org/10.1038/s41597-024-02931-y

#### Full workflow repository
- **Fareri trust-game workflow repo**: `https://github.com/tubric/fareri-2022-neuroimage`

#### Tools included in this workflow
- **fMRIPrep**: Esteban, O., Markiewicz, C. J., Blair, R. W., *et al.* (2019). fMRIPrep: a robust preprocessing pipeline for functional MRI. *Nature Methods, 16*, 111–116.
- **FSL / FEAT**: Jenkinson, M., Beckmann, C. F., Behrens, T. E. J., Woolrich, M. W., & Smith, S. M. (2012). FSL. *NeuroImage, 62*, 782–790.
- **DataLad**: Halchenko, Y. O., Meyer, K., Poldrack, B., *et al.* (2021). DataLad: distributed system for joint management of code, data, and their relationship. *Journal of Open Source Software, 6*, 3262.
- **bidsutils / BIDSto3col.sh**: Tom Nichols and contributors. `bids-standard/bidsutils`.
- **NiiVue / ipyniivue** for interactive image viewing in Jupyter.

#### Dataset
- **OpenNeuro ds003745** (trust game dataset used in class materials and labs)

#### Educational resources
- Course labs on Neurodesk, fMRIPrep, FEAT, and 3-column event files.


## Table of content
[1. Load software tools and import python libraries](#1.-Load-software-tools-and-import-python-libraries)  
[2. Data preparation](#2.-Data-preparation)  
[3. Run fMRIPrep for one subject](#3.-Run-fMRIPrep-for-one-subject)  
[4. Make a FEAT confounds file](#4.-Make-a-FEAT-confounds-file)  
[5. Convert events.tsv to 3-column files](#5.-Convert-events.tsv-to-3-column-files)  
[6. Render the FEAT template](#6.-Render-the-FEAT-template)  
[7. Run first-level FEAT](#7.-Run-first-level-FEAT)  
[8. Fix FEAT registration for fMRIPrep-preprocessed data](#8.-Fix-FEAT-registration-for-fMRIPrep-preprocessed-data)  
[9. Results](#9.-Results)  
[10. Dependencies in Jupyter/Python](#10.-Dependencies-in-Jupyter/Python)


## 1. Load software tools and import python libraries

In [ ]:
import module
await module.load('fmriprep/25.2.5')
await module.load('fsl/6.0.7.16')
await module.list()


In [ ]:
%%capture
!pip install pandas ipyniivue


In [ ]:
from pathlib import Path
import os
import glob
import pandas as pd
from IPython.display import display, Markdown, Image

base_dir = Path.home() / "trust_example"
print(base_dir)


## 2. Data preparation

We will keep everything for this example in one folder directly under the home directory:

`~/trust_example`

That makes the paths easy to read, and it also makes it easier to rerun or delete the whole example later if you want a clean start.

Inside that folder, we will make a few subdirectories:

- `templates/` for the FEAT template  
- `bids/` for the OpenNeuro dataset  
- `derivatives/` for fMRIPrep output, confounds, EV files, and FEAT output  
- `scratch/` for temporary working files  
- `bidsutils/` for Tom Nichols' `BIDSto3col.sh`


In [ ]:
%%bash
set -e

EXAMPLE_DIR="$HOME/trust_example"

mkdir -p "$EXAMPLE_DIR"
cd "$EXAMPLE_DIR"

# Keep the example organized in one project folder.
mkdir -p templates bids derivatives scratch

# Copy the one FEAT template we need.
curl -L \
  https://raw.githubusercontent.com/tubric/fareri-2022-neuroimage/main/templates/L1_task-trust_model-01_type-act.fsf \
  -o templates/L1_task-trust_model-01_type-act.fsf

# Install bidsutils separately so we can use BIDSto3col.sh.
if [ ! -d bidsutils ]; then
  git clone --depth 1 https://github.com/bids-standard/bidsutils.git
fi

# Install the OpenNeuro dataset skeleton and get the full subject recursively.
cd bids
if [ ! -d ds003745 ]; then
  datalad install https://github.com/OpenNeuroDatasets/ds003745.git
fi

cd ds003745
datalad get -r sub-104


In [ ]:
!tree -L 3 ~/trust_example/templates
!tree -L 3 ~/trust_example/bids/ds003745/sub-104


The example below uses:

- **subject**: `104`
- **task**: `trust`
- **run**: `01`

That keeps the notebook manageable, but the same logic can be repeated for the other runs.

For teaching purposes, one run is a nice compromise: the model is still realistic, but the files and paths stay simple enough to follow line by line.


## 3. Run fMRIPrep for one subject

The next cell runs fMRIPrep on one participant and writes the outputs under `~/trust_example/derivatives/fmriprep`.

A few practical notes:

- the FreeSurfer license file should exist at `~/.license`
- we turn off BIDS validation here because this dataset is known to be BIDS-compatible, but the validator is causing trouble in this environment
- this command will likely take **a couple of hours** to run
- if you are wondering whether it is still running, go back to the **Terminal** and type `top`
- the `antsRegistration` step is especially slow, so long stretches of apparent inactivity are normal
- we use `MNI152NLin6Asym:res-2` so the preprocessed BOLD file lands on a 2 mm MNI grid that is easy to use alongside standard FSL templates and atlases

One important detail: the `res-2` part controls the **output grid** of the resampled BOLD files. It does **not** change the resolution used for the underlying nonlinear normalization.


In [ ]:
!ls -l ~/.license


In [ ]:
%%bash
set -e

EXAMPLE_DIR="$HOME/trust_example"

sub=104
BIDS_DIR="$EXAMPLE_DIR/bids/ds003745"
OUT_DIR="$EXAMPLE_DIR/derivatives/fmriprep"
WORK_DIR="$EXAMPLE_DIR/scratch/fmriprep_work"
FS_LIC="$HOME/.license"

mkdir -p "$OUT_DIR" "$WORK_DIR"

export OMP_NUM_THREADS=1
export ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS=1

# This command will likely take a couple of hours to finish.
# If you want to confirm that it is still running, go back to the
# Terminal and type: top
# The antsRegistration process is especially slow and can run for
# a long time without printing much new output.

fmriprep \
  "$BIDS_DIR" \
  "$OUT_DIR" \
  participant \
  --participant-label "$sub" \
  --stop-on-first-crash \
  --skip-bids-validation \
  --fs-license-file "$FS_LIC" \
  --fs-no-reconall \
  --output-spaces MNI152NLin6Asym:res-2 \
  --nthreads 12 \
  --omp-nthreads 1 \
  --mem-mb 30000 \
  -w "$WORK_DIR"


When fMRIPrep finishes, the files we care about most for FEAT are:

- the HTML report
- the preprocessed BOLD file for the run we want
- the confounds table for that run

Because we wrote the outputs into `~/trust_example/derivatives/fmriprep`, the next steps can keep using the same paths throughout the notebook.

If the HTML report does not open correctly inside Neurodesktop, open it through **JupyterLab** instead. It is worth checking the report before moving on, because bad preprocessing or bad alignment will usually be easier to catch here than later in the FEAT output.


In [ ]:
!find ~/trust_example/derivatives/fmriprep -name "sub-104.html"
!find ~/trust_example/derivatives/fmriprep -name "*run-01*desc-preproc_bold.nii.gz"
!find ~/trust_example/derivatives/fmriprep -name "*run-01*desc-confounds_timeseries.tsv"


## 4. Make a FEAT confounds file

Here we take the fMRIPrep confounds table for one run and save a FEAT-ready text file.

We keep:
- cosine regressors
- non-steady-state regressors
- 6 motion parameters
- the first 6 aCompCor components
- framewise displacement, if it exists

FEAT wants a plain text file **without a header**, so we write the output that way.

This is a good example of why notebook workflows can be useful for teaching: you can see exactly which nuisance regressors are being kept, rather than hiding that logic inside a separate script.


In [ ]:
confounds_tsv = base_dir / "derivatives" / "fmriprep" / "sub-104" / "func" / "sub-104_task-trust_run-01_desc-confounds_timeseries.tsv"
confounds = pd.read_csv(confounds_tsv, sep="	")

# Keep a simple but still realistic set of nuisance regressors.
cosine_cols = [c for c in confounds.columns if c.startswith("cosine")]
nss_cols = [c for c in confounds.columns if c.startswith("non_steady_state")]
motion_cols = ["trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"]
acompcor_cols = [c for c in confounds.columns if c.startswith("a_comp_cor_")][:6]
fd_cols = ["framewise_displacement"] if "framewise_displacement" in confounds.columns else []

keep_cols = cosine_cols + nss_cols + motion_cols + acompcor_cols + fd_cols
feat_confounds = confounds[keep_cols].fillna(0)

out_dir = base_dir / "derivatives" / "fsl" / "confounds" / "sub-104"
out_dir.mkdir(parents=True, exist_ok=True)

out_file = out_dir / "sub-104_task-trust_run-01_desc-fslConfounds.tsv"
feat_confounds.to_csv(out_file, sep="	", header=False, index=False)

print(out_file)
print(feat_confounds.shape)
feat_confounds.head()


## 5. Convert events.tsv to 3-column files

This step uses `BIDSto3col.sh` from **bidsutils** and keeps the logic to one run.

`BIDSto3col.sh` reads the BIDS events file and writes one 3-column text file per event type.  
Each output file has three columns:

1. onset  
2. duration  
3. weight

Those output files are exactly the kind of timing files FEAT expects for custom EVs, so this is one of the main bridges between a BIDS dataset and an FSL first-level model.


In [ ]:
%%bash
set -e

EXAMPLE_DIR="$HOME/trust_example"

sub=104
run=01
events_tsv="$EXAMPLE_DIR/bids/ds003745/sub-${sub}/func/sub-${sub}_task-trust_run-${run}_events.tsv"
out_base="$EXAMPLE_DIR/derivatives/fsl/EVfiles/sub-${sub}/trust/run-${run}"

mkdir -p "$(dirname "$out_base")"

# This creates one 3-column file per event type found in the BIDS events file.
bash "$EXAMPLE_DIR/bidsutils/BIDSto3col/BIDSto3col.sh" "$events_tsv" "$out_base"


In [ ]:
!ls ~/trust_example/derivatives/fsl/EVfiles/sub-104/trust
!head -n 5 ~/trust_example/derivatives/fsl/EVfiles/sub-104/trust/run-01_choice_computer.txt


## 6. Render the FEAT template

The template file already contains the FEAT model structure.  
What changes from subject to subject and run to run are the **paths** and a few settings.

This is a useful pattern for reproducible work: keep the design mostly fixed, and then fill in the parts that should vary across runs.

### Placeholders we replace
- `OUTPUT` → where the `.feat` directory should be written  
- `DATA` → the preprocessed BOLD file from fMRIPrep  
- `EVDIR` → the prefix used by the EV timing files  
- `MISSED_TRIAL` → path to the optional missed-trial EV file  
- `EV_SHAPE` → `3` if the missed-trial file exists, otherwise `10` for an empty EV  
- `SMOOTH` → smoothing kernel in mm  
- `CONFOUNDEVS` → the confounds file we just created


In [ ]:
template_text = (base_dir / "templates" / "L1_task-trust_model-01_type-act.fsf").read_text()

for token in ["OUTPUT", "DATA", "EVDIR", "MISSED_TRIAL", "EV_SHAPE", "SMOOTH", "CONFOUNDEVS"]:
    print(f"--- lines containing {token} ---")
    for line in template_text.splitlines():
        if token in line:
            print(line)
    print()


The next cell renders a new `.fsf` file. Each variable is written out explicitly so you can see what is being inserted.

Even if you do not remember every line of FEAT syntax, the important lesson is that the `.fsf` file is just text. That means you can inspect it, edit it, and generate it systematically.


In [ ]:
%%bash
set -e

EXAMPLE_DIR="$HOME/trust_example"

TASK=trust
SMOOTH=6
sub=104
run=01

OUTPUT="$EXAMPLE_DIR/derivatives/fsl/sub-${sub}/L1_task-${TASK}_model-01_type-act_run-${run}_sm-${SMOOTH}"
DATA="$EXAMPLE_DIR/derivatives/fmriprep/sub-${sub}/func/sub-${sub}_task-${TASK}_run-${run}_space-MNI152NLin6Asym_res-2_desc-preproc_bold.nii.gz"
CONFOUNDEVS="$EXAMPLE_DIR/derivatives/fsl/confounds/sub-${sub}/sub-${sub}_task-${TASK}_run-${run}_desc-fslConfounds.tsv"
EVDIR="$EXAMPLE_DIR/derivatives/fsl/EVfiles/sub-${sub}/${TASK}/run-${run}"
MISSED_TRIAL="${EVDIR}_missed_trial.txt"

if [ -e "$MISSED_TRIAL" ]; then
  EV_SHAPE=3
else
  EV_SHAPE=10
fi

mkdir -p "$(dirname "$OUTPUT")"

sed \
  -e "s@OUTPUT@${OUTPUT}@g" \
  -e "s@DATA@${DATA}@g" \
  -e "s@EVDIR@${EVDIR}@g" \
  -e "s@MISSED_TRIAL@${MISSED_TRIAL}@g" \
  -e "s@EV_SHAPE@${EV_SHAPE}@g" \
  -e "s@SMOOTH@${SMOOTH}@g" \
  -e "s@CONFOUNDEVS@${CONFOUNDEVS}@g" \
  "$EXAMPLE_DIR/templates/L1_task-trust_model-01_type-act.fsf" \
  > "$EXAMPLE_DIR/derivatives/fsl/sub-${sub}/L1_sub-${sub}_task-${TASK}_model-01_run-${run}_act.fsf"


In [ ]:
rendered_fsf = base_dir / "derivatives" / "fsl" / "sub-104" / "L1_sub-104_task-trust_model-01_run-01_act.fsf"
print(rendered_fsf)
print()
print(rendered_fsf.read_text()[:4000])


## 7. Run first-level FEAT

This command runs FEAT on the rendered design file.

At this point, the main conceptual pieces are in place: preprocessed data, confounds, EV timing files, and a rendered design file. FEAT now uses those pieces to estimate the first-level GLM.


In [ ]:
%%bash
set -e

# Run FEAT on the rendered first-level design file.
feat "$HOME/trust_example/derivatives/fsl/sub-104/L1_sub-104_task-trust_model-01_run-01_act.fsf"


## 8. Fix FEAT registration for fMRIPrep-preprocessed data

Because the functional image already came out of fMRIPrep in standard space, FEAT's usual registration outputs are not the right ones to trust here.  
The next cell follows the same post-FEAT registration fix used in the original `L1stats.sh` script.

This is based on the NeuroStars note referenced in that script:  
https://neurostars.org/t/performing-full-glm-analysis-with-fsl-on-the-bold-images-preprocessed-by-fmriprep-without-re-registering-the-data-to-the-mni-space/784/3

In plain language, this step tells FEAT to treat the input as already being in standard space, so the registration files inside the `.feat` directory do not point to misleading transforms.


In [ ]:
%%bash
set -e

OUTPUT="$HOME/trust_example/derivatives/fsl/sub-104/L1_task-trust_model-01_type-act_run-01_sm-6"

# fix registration as per NeuroStars post:
# https://neurostars.org/t/performing-full-glm-analysis-with-fsl-on-the-bold-images-preprocessed-by-fmriprep-without-re-registering-the-data-to-the-mni-space/784/3
mkdir -p ${OUTPUT}.feat/reg
ln -sf $FSLDIR/etc/flirtsch/ident.mat ${OUTPUT}.feat/reg/example_func2standard.mat
ln -sf $FSLDIR/etc/flirtsch/ident.mat ${OUTPUT}.feat/reg/standard2example_func.mat
ln -sf ${OUTPUT}.feat/mean_func.nii.gz ${OUTPUT}.feat/reg/standard.nii.gz


## 9. Results

The FEAT output directory should now contain the standard report, design matrix, and statistical maps.

A few files worth checking first are:

- `report.html`
- `design.png`
- `thresh_zstat1.nii.gz`
- `thresh_zstat2.nii.gz`
- `thresh_zstat10.nii.gz`

This is also a good point to slow down and inspect the outputs before interpreting anything. Make sure the design looks sensible, the report opens, and the expected thresholded maps actually exist.


In [ ]:
feat_dir = base_dir / "derivatives" / "fsl" / "sub-104" / "L1_task-trust_model-01_type-act_run-01_sm-6.feat"

for rel in [
    "report.html",
    "design.png",
    "thresh_zstat1.nii.gz",
    "thresh_zstat2.nii.gz",
    "thresh_zstat10.nii.gz",
]:
    p = feat_dir / rel
    print(f"{p}: {p.exists()}")


In [ ]:
design_png = feat_dir / "design.png"
if design_png.exists():
    display(Image(filename=str(design_png)))
else:
    print("Run FEAT first, then come back to this cell.")


### Optional: try a quick NiiVue visualization

This cell overlays a thresholded FEAT result on the example functional image from the same run.

Here we use `thresh_zstat10.nii.gz`, which corresponds to **reciprocate > defect**. In this task, that contrast is roughly reward-related, so even in a single run you may see activation in the **ventral striatum** and nearby reward-sensitive regions.

A few learning notes:

- `example_func.nii.gz` gives you a background image in the same space as the FEAT results
- `thresh_zstat10.nii.gz` is already thresholded, so the overlay is easier to interpret
- if your notebook shows nothing, first confirm that the `.feat` directory exists and that `thresh_zstat10.nii.gz` is really there


In [ ]:
from pathlib import Path
from IPython.display import display
from ipyniivue import NiiVue

# Use the example functional image as the background.
# This keeps the overlay and background in the same space and grid.
bg = str(feat_dir / "example_func.nii.gz")
zmap = str(feat_dir / "thresh_zstat10.nii.gz")  # reciprocate > defect

print("Background exists:", Path(bg).exists(), bg)
print("Contrast exists:", Path(zmap).exists(), zmap)

if Path(zmap).exists():
    nv = NiiVue()
    nv.load_volumes([
        {"path": bg, "name": "example_func", "colormap": "gray", "opacity": 1.0},
        {
            "path": zmap,
            "name": "reciprocate > defect",
            "colormap": "red",
            "opacity": 0.6
        }
    ])
    display(nv)
else:
    print("Run FEAT first, then come back to this cell.")


## 10. Dependencies in Jupyter/Python
- Using the package [watermark](https://github.com/rasbt/watermark) to document system environment and software versions used in this notebook, alongside the Neurodesktop version extracted from the `JUPYTER_IMAGE` environment variable.


In [ ]:
import os

%load_ext watermark

%watermark
%watermark --iversions

neurodesktop_version = os.environ.get('JUPYTER_IMAGE', 'unknown').split(':')[-1]
print(f"Neurodesktop version: {neurodesktop_version}")
